# NST (New Solar Telescope) — Performance Analysis & MCAO Simulation
# NST (New Solar Telescope) — 성능 분석 및 MCAO 시뮬레이션

**Paper**: Goode & Cao, "The 1.6 m Off-Axis New Solar Telescope (NST) in Big Bear," *Proc. SPIE* 8444, 844403 (2012)

이 노트북은 논문에서 다루는 핵심 개념들을 수치적으로 재현합니다:

This notebook numerically reproduces key concepts from the paper:

1. **회절 한계 분해능 비교** — NST vs SST vs DKIST / Diffraction limit comparison
2. **AO 모드 수 분석** — 왜 AO-76이 부족하고 AO-308이 필요한가 / Why AO-76 is insufficient
3. **Strehl ratio vs 파장** — AO-76과 AO-308의 파장별 성능 / Wavelength-dependent Strehl
4. **중앙 차폐가 PSF/MTF에 미치는 영향** — Off-axis의 이점 / Off-axis advantage
5. **MCAO 시뮬레이션** — 다중 DM 공역의 효과 / Multi-conjugate DM effect

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import j1

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3
})

## 1. 회절 한계 분해능 비교 / Diffraction-Limited Resolution Comparison

회절 한계 분해능은 $\theta = 1.22 \lambda / D$ 로 주어집니다.
NST(1.6m), SST(1.0m), DKIST(4.0m)의 파장별 분해능을 비교합니다.

The diffraction-limited resolution is given by $\theta = 1.22 \lambda / D$.
We compare resolution across wavelength for NST (1.6 m), SST (1.0 m), and DKIST (4.0 m).

In [ ]:
# Telescope parameters
telescopes = {
    'SST (1.0 m)': 1.0,
    'NST (1.6 m)': 1.6,
    'DKIST (4.0 m)': 4.0
}
colors = {'SST (1.0 m)': 'C0', 'NST (1.6 m)': 'C1', 'DKIST (4.0 m)': 'C2'}

wavelengths = np.linspace(0.4, 1.7, 200)  # um
RAD_TO_ARCSEC = 206265

fig, ax = plt.subplots(figsize=(10, 6))

for name, D in telescopes.items():
    theta = 1.22 * (wavelengths * 1e-6) / D * RAD_TO_ARCSEC
    ax.plot(wavelengths, theta, label=name, color=colors[name], linewidth=2)

# Mark key wavelengths from the paper
key_wavelengths = {
    'TiO (705.7 nm)': 0.7057,
    r'H$\alpha$ (656.3 nm)': 0.6563,
    'Fe I (1565 nm)': 1.565,
    'G-band (430 nm)': 0.430
}

for label, wl in key_wavelengths.items():
    ax.axvline(wl, color='gray', linestyle=':', alpha=0.5)
    ax.annotate(label, (wl, ax.get_ylim()[1]*0.02), rotation=90,
                fontsize=8, va='bottom', ha='right')

ax.set_xlabel('Wavelength (μm) / 파장')
ax.set_ylabel('Diffraction Limit (arcsec) / 회절 한계')
ax.set_title('Diffraction-Limited Resolution: NST vs SST vs DKIST\n회절 한계 분해능 비교')
ax.legend()
ax.set_xlim(0.4, 1.7)
plt.tight_layout()
plt.show()

# Print specific values from the paper
print('Diffraction limit at key wavelengths / 주요 파장에서의 회절 한계:')
print(f'{"":20s} {"SST (1.0m)":>12s} {"NST (1.6m)":>12s} {"DKIST (4.0m)":>12s}')
for label, wl in key_wavelengths.items():
    vals = []
    for name, D in telescopes.items():
        theta = 1.22 * (wl * 1e-6) / D * RAD_TO_ARCSEC
        vals.append(f'{theta:.3f}\"')
    print(f'{label:20s} {vals[0]:>12s} {vals[1]:>12s} {vals[2]:>12s}')

## 2. AO 모드 수 분석 / AO Mode Count Analysis

효과적인 AO 보정에 필요한 Zernike 모드 수: $N_{\text{modes}} \approx 0.97 (D/r_0)^2$

논문에서 AO-76이 가시광에서 부족하고 AO-308이 필요한 이유를 정량적으로 확인합니다.

Number of Zernike modes needed: $N_{\text{modes}} \approx 0.97 (D/r_0)^2$.
We quantitatively verify why AO-76 is insufficient in visible and AO-308 is needed.

In [ ]:
D = 1.6  # NST aperture in meters

# r0 values at 500nm from Kellerer et al. (2012)
r0_range = np.linspace(0.03, 0.20, 200)  # meters

N_modes = 0.97 * (D / r0_range) ** 2

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(r0_range * 100, N_modes, 'k-', linewidth=2, label='Required modes / 필요 모드 수')

# AO system capacities
ao_systems = {
    'AO-76 (76 sub-apertures)': 76,
    'AO-308 (308 sub-apertures)': 308
}

for name, n_sub in ao_systems.items():
    ax.axhline(n_sub, linestyle='--', alpha=0.7, label=name)

# Mark BBSO seeing conditions
r0_median_summer = 9.1  # cm (Kellerer+2012, summer)
r0_median_winter = 5.5  # cm (winter)
r0_steady = 6.0         # cm (steady state, from paper)

for r0_val, label, color in [
    (r0_median_winter, f'Winter median\n$r_0$={r0_median_winter} cm', 'blue'),
    (r0_steady, f'Steady state\n$r_0$={r0_steady} cm', 'purple'),
    (r0_median_summer, f'Summer median\n$r_0$={r0_median_summer} cm', 'red')
]:
    N_req = 0.97 * (D / (r0_val / 100)) ** 2
    ax.plot(r0_val, N_req, 'o', color=color, markersize=10, zorder=5)
    ax.annotate(label, (r0_val, N_req), textcoords='offset points',
                xytext=(15, 10), fontsize=9, color=color)

ax.set_xlabel('Fried parameter $r_0$ at 500 nm (cm)')
ax.set_ylabel('Required Zernike modes / 필요 Zernike 모드 수')
ax.set_title('AO Mode Requirements for NST (1.6 m)\nNST의 AO 모드 요구 사항')
ax.legend(loc='upper right')
ax.set_xlim(3, 20)
ax.set_ylim(10, 5000)
plt.tight_layout()
plt.show()

# Summary table
print('\nAO adequacy at BBSO seeing conditions / BBSO 시상에서의 AO 적합성:')
print(f'{"Condition":20s} {"r0 (cm)":>8s} {"N_modes":>8s} {"AO-76":>8s} {"AO-308":>8s}')
for cond, r0 in [('Winter median', 5.5), ('Steady (paper)', 6.0), ('Summer median', 9.1), ('Excellent', 15.0)]:
    N = 0.97 * (D / (r0 / 100)) ** 2
    ao76 = '✓' if N <= 76 else '✗'
    ao308 = '✓' if N <= 308 else '✗'
    print(f'{cond:20s} {r0:8.1f} {N:8.0f} {ao76:>8s} {ao308:>8s}')

## 3. Strehl Ratio vs 파장 / Strehl Ratio vs Wavelength

Maréchal 근사: $S \approx \exp(-\sigma_\phi^2)$, 여기서 $\sigma_\phi$는 잔여 wavefront error (radian).

AO 보정 후 잔여 fitting error: $\sigma_{\text{fit}}^2 \approx 0.286 (d/r_0)^{5/3}$ (rad²)
여기서 $d$는 서브-애퍼처 간격, $r_0(\lambda) = r_0(\lambda_0)(\lambda/\lambda_0)^{6/5}$.

Maréchal approximation: $S \approx \exp(-\sigma_\phi^2)$.
Residual fitting error after AO: $\sigma_{\text{fit}}^2 \approx 0.286 (d/r_0)^{5/3}$ (rad²),
where $d$ is sub-aperture spacing and $r_0(\lambda) = r_0(\lambda_0)(\lambda/\lambda_0)^{6/5}$.

In [ ]:
def strehl_vs_wavelength(D, n_sub, r0_500, wavelengths_um):
    """Calculate Strehl ratio vs wavelength for an AO system.

    Args:
        D: Telescope diameter in meters.
        n_sub: Number of sub-apertures across diameter.
        r0_500: Fried parameter at 500 nm in meters.
        wavelengths_um: Array of wavelengths in micrometers.

    Returns:
        Array of Strehl ratios.
    """
    d = D / np.sqrt(n_sub)  # Sub-aperture spacing
    lambda_ref = 0.5e-6  # Reference wavelength (500 nm)
    wavelengths_m = wavelengths_um * 1e-6

    # r0 scales as lambda^(6/5)
    r0 = r0_500 * (wavelengths_m / lambda_ref) ** (6/5)

    # Fitting error (Noll, 1976): sigma^2 = 0.286 * (d/r0)^(5/3) in rad^2
    # Convert to phase variance at each wavelength
    sigma_fit_rad2 = 0.286 * (d / r0) ** (5/3)

    # Additional terms: temporal error, noise, etc. (simplified)
    # Add ~30% overhead for other error sources
    sigma_total_rad2 = sigma_fit_rad2 * 1.3

    strehl = np.exp(-sigma_total_rad2)
    return np.clip(strehl, 0, 1)


wavelengths = np.linspace(0.4, 1.7, 300)
r0_500 = 0.06  # 6 cm at 500nm (steady state, from paper)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: median seeing
ax = axes[0]
for label, n_sub, ls in [('AO-76', 76, '--'), ('AO-308', 308, '-')]:
    S = strehl_vs_wavelength(1.6, n_sub, 0.06, wavelengths)
    ax.plot(wavelengths, S, ls, linewidth=2, label=label)

ax.axhline(0.8, color='red', linestyle=':', alpha=0.5, label='Diffraction limit (S=0.8)')
ax.set_xlabel('Wavelength (μm)')
ax.set_ylabel('Strehl Ratio')
ax.set_title('Median seeing ($r_0$ = 6 cm)\n중간 시상')
ax.legend()
ax.set_ylim(0, 1)

# Right: good summer seeing
ax = axes[1]
for label, n_sub, ls in [('AO-76', 76, '--'), ('AO-308', 308, '-')]:
    S = strehl_vs_wavelength(1.6, n_sub, 0.10, wavelengths)
    ax.plot(wavelengths, S, ls, linewidth=2, label=label)

ax.axhline(0.8, color='red', linestyle=':', alpha=0.5, label='Diffraction limit (S=0.8)')
ax.set_xlabel('Wavelength (μm)')
ax.set_ylabel('Strehl Ratio')
ax.set_title('Good summer seeing ($r_0$ = 10 cm)\n좋은 여름 시상')
ax.legend()
ax.set_ylim(0, 1)

fig.suptitle('NST Strehl Ratio: AO-76 vs AO-308\nNST Strehl 비율: AO-76 vs AO-308',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Print paper-consistent values
print('Strehl at key wavelengths (r0=6cm) / 주요 파장에서의 Strehl (r0=6cm):')
for wl_um, name in [(0.5, '500 nm (visible)'), (1.0, '1.0 μm (NIR)'), (1.565, '1565 nm (IRIM)')]:
    s76 = strehl_vs_wavelength(1.6, 76, 0.06, np.array([wl_um]))[0]
    s308 = strehl_vs_wavelength(1.6, 308, 0.06, np.array([wl_um]))[0]
    print(f'  {name:20s}: AO-76 S={s76:.2f}, AO-308 S={s308:.2f}')

## 4. 중앙 차폐가 PSF와 MTF에 미치는 영향 / Central Obscuration Effect on PSF and MTF

Off-axis 설계의 핵심 이점은 중앙 차폐(central obscuration) 제거입니다.
차폐비 $\epsilon$에 따른 PSF와 MTF 변화를 비교합니다.

The key advantage of off-axis design is elimination of central obscuration.
We compare PSF and MTF changes with obscuration ratio $\epsilon$.

In [ ]:
def annular_psf(theta, epsilon=0):
    """Compute PSF for an annular aperture with obscuration ratio epsilon.

    Args:
        theta: Normalized angle (pi * D * angle / lambda).
        epsilon: Obscuration ratio (0 = no obscuration, 1 = fully blocked).

    Returns:
        Normalized PSF intensity.
    """
    # Avoid division by zero
    theta = np.where(np.abs(theta) < 1e-10, 1e-10, theta)

    airy_outer = 2 * j1(theta) / theta
    if epsilon == 0:
        return airy_outer ** 2

    airy_inner = 2 * j1(epsilon * theta) / (epsilon * theta)
    psf = (airy_outer - epsilon**2 * airy_inner) ** 2 / (1 - epsilon**2) ** 2
    return psf


def annular_mtf(freq, epsilon=0):
    """Compute MTF for an annular aperture (approximate).

    Args:
        freq: Normalized spatial frequency (0 to 1, where 1 = D/lambda).
        epsilon: Obscuration ratio.

    Returns:
        MTF value.
    """
    # Unobscured circular aperture OTF
    f = np.clip(freq, 0, 1)
    otf_full = (2/np.pi) * (np.arccos(f) - f * np.sqrt(1 - f**2))

    if epsilon == 0:
        return otf_full

    # Approximate annular OTF: subtract inner circle contribution
    f_inner = np.clip(freq / epsilon, 0, 1)
    otf_inner = (2/np.pi) * (np.arccos(f_inner) - f_inner * np.sqrt(1 - f_inner**2))
    otf_inner *= epsilon ** 2

    otf_annular = (otf_full - otf_inner) / (1 - epsilon**2)
    return np.clip(otf_annular, 0, 1)


# Obscuration ratios to compare
epsilons = {
    'Off-axis (ε=0, NST/SST)': 0.0,
    'Small obs. (ε=0.2)': 0.2,
    'Typical Cassegrain (ε=0.33)': 0.33,
    'Large obs. (ε=0.5)': 0.5
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PSF comparison
ax = axes[0]
theta = np.linspace(-15, 15, 1000)
for label, eps in epsilons.items():
    psf = annular_psf(theta, eps)
    ax.semilogy(theta / np.pi, psf, linewidth=1.5, label=label)

ax.set_xlabel('Angle ($\\lambda/D$ units)')
ax.set_ylabel('Normalized Intensity (log)')
ax.set_title('PSF Comparison / PSF 비교')
ax.set_ylim(1e-5, 1.2)
ax.legend(fontsize=9)

# MTF comparison
ax = axes[1]
freq = np.linspace(0, 1, 500)
for label, eps in epsilons.items():
    mtf = annular_mtf(freq, eps)
    ax.plot(freq, mtf, linewidth=1.5, label=label)

ax.set_xlabel('Normalized Spatial Frequency ($f / f_{\\mathrm{cutoff}}$)')
ax.set_ylabel('MTF')
ax.set_title('MTF Comparison / MTF 비교')
ax.legend(fontsize=9)

fig.suptitle('Effect of Central Obscuration on PSF and MTF\n중앙 차폐가 PSF와 MTF에 미치는 영향',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Quantify contrast loss
print('\nPSF first ring intensity (contrast loss) / PSF 첫 번째 링 강도:')
theta_first_ring = 5.14  # First Airy ring location
for label, eps in epsilons.items():
    ring = annular_psf(theta_first_ring, eps)
    print(f'  {label:35s}: {ring:.5f} ({ring/annular_psf(theta_first_ring, 0):.1f}× vs unobscured)')

## 5. MCAO 시뮬레이션 — Isoplanatic Angle과 보정 시야 / MCAO Simulation

단일 DM AO는 isoplanatic angle $\theta_0$ 내에서만 유효합니다.
MCAO는 여러 고도의 난류층을 각각 보정하여 보정 시야를 확대합니다.

논문 Fig. 3의 개념을 간략한 모델로 재현합니다.

Single-DM AO is effective only within isoplanatic angle $\theta_0$.
MCAO corrects turbulence at multiple altitudes, expanding the corrected FOV.
We reproduce the concept of Fig. 3 with a simplified model.

In [ ]:
def isoplanatic_angle_mcao(h_layers, cn2_fractions, r0_total, h_dms, wavelength_um=0.5):
    """Estimate effective isoplanatic angle with MCAO correction.

    Computes the isoplanatic angle considering which turbulence layers
    are corrected by conjugated DMs.

    Args:
        h_layers: Turbulence layer altitudes in meters.
        cn2_fractions: Fraction of total Cn2 at each layer (sums to 1).
        r0_total: Total Fried parameter in meters.
        h_dms: DM conjugation altitudes in meters (empty for no AO).
        wavelength_um: Wavelength in micrometers.

    Returns:
        Effective isoplanatic angle in arcseconds.
    """
    wavelength_m = wavelength_um * 1e-6

    # For each layer, find the closest DM
    residual_cn2 = np.zeros_like(cn2_fractions)
    for i, h in enumerate(h_layers):
        if len(h_dms) == 0:
            residual_cn2[i] = cn2_fractions[i]
        else:
            # Residual is proportional to distance from nearest DM
            distances = np.abs(h - np.array(h_dms))
            min_dist = np.min(distances)
            # Simple model: correction effectiveness decreases with distance
            correction = np.exp(-(min_dist / 2000) ** 2)  # 2km scale
            residual_cn2[i] = cn2_fractions[i] * (1 - correction)

    # Effective h_bar for residual turbulence
    if np.sum(residual_cn2) < 1e-10:
        return 100.0  # Effectively infinite

    h_eff = (np.sum(residual_cn2 * h_layers ** (5/3)) / np.sum(residual_cn2)) ** (3/5)

    # Residual r0
    r0_residual = r0_total / np.sum(residual_cn2) ** (3/5)

    # Isoplanatic angle: theta_0 = 0.314 * r0 / h_eff
    theta_0 = 0.314 * r0_residual / max(h_eff, 1) * RAD_TO_ARCSEC

    return min(theta_0, 100.0)


# Atmospheric model based on paper discussion
# Turbulence layers: ground, boundary layer, free atmosphere, tropopause
h_layers = np.array([0, 500, 1000, 3000, 6000, 10000])  # meters

# Summer-like Cn2 profile (ground-dominant)
cn2_summer = np.array([0.40, 0.15, 0.10, 0.15, 0.10, 0.10])

# Winter-like Cn2 profile (more distributed)
cn2_winter = np.array([0.25, 0.10, 0.10, 0.15, 0.20, 0.20])

r0_summer = 0.091  # 9.1 cm
r0_winter = 0.055  # 5.5 cm

# DM configurations from paper
dm_configs = {
    'No AO': [],
    'Ground-layer AO': [0],
    '2-DM MCAO (0+3km)': [0, 3000],
    '3-DM MCAO (0+3+6km)': [0, 3000, 6000]
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, cn2, r0, season in [(axes[0], cn2_summer, r0_summer, 'Summer / 여름'),
                             (axes[1], cn2_winter, r0_winter, 'Winter / 겨울')]:
    configs = list(dm_configs.keys())
    angles = [isoplanatic_angle_mcao(h_layers, cn2, r0, dms)
              for dms in dm_configs.values()]

    bars = ax.barh(configs, angles, color=['gray', 'C0', 'C1', 'C2'])
    ax.set_xlabel('Effective Isoplanatic Angle (arcsec)')
    ax.set_title(f'{season}\n($r_0$ = {r0*100:.1f} cm)')

    for bar, angle in zip(bars, angles):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{angle:.1f}"', va='center', fontsize=10)

    # Mark NST FOV
    ax.axvline(35, color='red', linestyle=':', alpha=0.5)
    ax.text(35, -0.3, 'NST FOV\nhalf-width\n35"', fontsize=8, color='red', ha='center')

fig.suptitle('MCAO: Effective Isoplanatic Angle by Configuration\n'
             'MCAO: 구성별 유효 등방향각',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Zeeman Splitting — NIR의 이점 / Zeeman Splitting — NIR Advantage

IRIM이 1565 nm에서 운용하는 물리적 이유를 정량적으로 확인합니다.
Zeeman splitting: $\Delta\lambda = \frac{e \lambda^2 g B}{4\pi m_e c}$

We quantitatively verify the physical reason IRIM operates at 1565 nm.

In [ ]:
# Physical constants
e = 1.602e-19       # C
m_e = 9.109e-31     # kg
c = 2.998e8          # m/s

def zeeman_splitting(wavelength_nm, g_eff, B_gauss):
    """Calculate Zeeman splitting in pm.

    Args:
        wavelength_nm: Wavelength in nm.
        g_eff: Effective Lande g-factor.
        B_gauss: Magnetic field strength in Gauss.

    Returns:
        Zeeman splitting in pm.
    """
    lam_m = wavelength_nm * 1e-9
    B_tesla = B_gauss * 1e-4
    delta_lambda = e * lam_m**2 * g_eff * B_tesla / (4 * np.pi * m_e * c)
    return delta_lambda * 1e12  # Convert to pm


# Spectral lines used by NST instruments
lines = {
    'Fe I 6302 Å (VIM)': (630.2, 2.5),
    'Fe I 15648 Å (IRIM)': (1564.8, 3.0),
    'He I 10830 Å (NIRIS)': (1083.0, 1.0),
    'Ca II 8542 Å (FISS)': (854.2, 1.1)
}

B_fields = np.linspace(10, 3000, 200)  # Gauss

fig, ax = plt.subplots(figsize=(10, 6))

for label, (wl, g) in lines.items():
    splitting = zeeman_splitting(wl, g, B_fields)
    ax.plot(B_fields, splitting, linewidth=2, label=label)

# Mark typical field strengths
for B, name in [(100, 'Quiet Sun'), (500, 'Plage'), (2000, 'Sunspot')]:
    ax.axvline(B, color='gray', linestyle=':', alpha=0.4)
    ax.text(B, ax.get_ylim()[1]*0.95 if B == 100 else ax.get_ylim()[1]*0.01,
            name, fontsize=9, ha='center', color='gray')

ax.set_xlabel('Magnetic Field (Gauss) / 자기장')
ax.set_ylabel('Zeeman Splitting (pm) / 제만 분리')
ax.set_title('Zeeman Splitting vs Magnetic Field by Spectral Line\n'
             '스펙트럼 선별 제만 분리 vs 자기장')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

# Print advantage ratio
print('\nZeeman splitting advantage of Fe I 15648 over Fe I 6302 / NIR 이점:')
for B in [100, 500, 2000]:
    s_nir = zeeman_splitting(1564.8, 3.0, B)
    s_vis = zeeman_splitting(630.2, 2.5, B)
    print(f'  B={B:4d} G: NIR={s_nir:7.1f} pm, Vis={s_vis:6.1f} pm, ratio={s_nir/s_vis:.1f}×')

## 요약 / Summary

이 노트북에서 재현한 핵심 결과:

Key results reproduced in this notebook:

1. **NST(1.6m)는 가시광에서 0.08" 회절 한계** — SST(1.0m)의 1.6배 개선
   NST achieves 0.08" diffraction limit in visible — 1.6× improvement over SST

2. **AO-76은 가시광에서 불충분** — 중간 시상에서 필요 모드 ~690, AO-76은 76개만 보정
   AO-76 insufficient in visible — ~690 modes needed at median seeing, AO-76 corrects only 76

3. **Off-axis 설계의 PSF/MTF 우위** — 중앙 차폐 제거로 고주파 대비가 크게 개선
   Off-axis PSF/MTF advantage — contrast greatly improved at high spatial frequencies

4. **MCAO로 보정 시야 확대** — 3-DM 구성이 가장 효과적이나, ground-layer 보정이 가장 큰 기여
   MCAO expands corrected FOV — 3-DM most effective, but ground-layer correction contributes most

5. **NIR에서 Zeeman splitting이 ~7.5배** — IRIM의 1565nm 선택은 약한 자기장 검출에 필수적
   Zeeman splitting ~7.5× larger in NIR — IRIM's 1565 nm choice essential for weak field detection